In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l1.3 Softmax

Softmax turns raw scores (logits) into a probability distribution: exponentiate,
then normalize. Build it by hand, then check it against PyTorch.

In [ ]:
from torch.nn import functional as F

logits = torch.tensor([2.0, 1.0, 0.1])
by_hand = logits.exp() / logits.exp().sum()
builtin = F.softmax(logits, dim=-1)
print("by hand: ", by_hand)
print("F.softmax:", builtin)

Both agree, and the result sums to 1: it is a valid distribution over the
next token.

In [ ]:
assert torch.allclose(by_hand, builtin)
assert abs(builtin.sum().item() - 1.0) < 1e-6